In [ ]:
# Setup directory structure
import os

# Create directories
os.makedirs('my_model', exist_ok=True)
os.makedirs('my_model/code', exist_ok=True)

# Move model file
# Assuming your model.pth is in the current directory
if os.path.exists('model.pth'):
    os.rename('model.pth', 'my_model/model.pth')
    
print("done")

In [ ]:
# Quick directory structure verification
def verify_structure():
    required_files = [
        'my_model/model.pth',
        'my_model/code/inference.py',
        'my_model/code/requirements.txt'
    ]
    
    for file_path in required_files:
        if not os.path.exists(file_path):
            print(f"Missing: {file_path}")
            return False
    return True

# Run verification
if verify_structure():
    print("Directory structure is correct")
else:
    print("Please fix the directory structure before proceeding")

In [ ]:
def create_model_archive():
    print("Creating model archive...")
    
    # Verify directory structure
    if not os.path.exists('my_model'):
        raise FileNotFoundError("my_model directory not found")
    if not os.path.exists('my_model/code'):
        raise FileNotFoundError("my_model/code directory not found")
    if not os.path.exists('my_model/model.pth'):
        raise FileNotFoundError("model.pth not found in my_model directory")
    if not os.path.exists('my_model/code/inference.py'):
        raise FileNotFoundError("inference.py not found in my_model/code directory")
    
    # Create archive
    with tarfile.open('model.tar.gz', 'w:gz') as tar:
        tar.add('my_model', arcname='.')
    
    # Verify archive contents
    with tarfile.open('model.tar.gz', 'r:gz') as tar:
        files = tar.getnames()
        print("\nArchive contents:")
        for f in files:
            print(f"- {f}")
    
    print("Model archive created successfully")

In [114]:
# Create model archive
create_model_archive()

Creating model archive...

Archive contents:
- .
- ./code
- ./code/inference.py
- ./code/requirements.txt
- ./model.pth
Model archive created successfully


In [124]:
def deploy_serverless_model():
    """Deploy model using serverless endpoint"""
    try:
        # Initialize sessions
        boto_session = boto3.Session(region_name=REGION)
        sagemaker_client = boto_session.client('sagemaker')
        s3_client = boto_session.client('s3')
        session = sagemaker.Session(
            boto_session=boto_session,
            sagemaker_client=sagemaker_client
        )
        bucket = session.default_bucket()

        print(f"\nStarting serverless deployment at {TIMESTAMP} UTC")
        print(f"User: {USER}")
        print(f"Role ARN: {ROLE_ARN}")
        print(f"Region: {REGION}")
        print(f"S3 Bucket: {bucket}")

        # Prepare and upload model
        prepare_model_package()
        
        print("\nUploading model to S3...")
        model_key = f'models/brain-ct/model-{int(datetime.utcnow().timestamp())}.tar.gz'
        with open('model.tar.gz', 'rb') as f:
            s3_client.upload_fileobj(
                f,
                bucket,
                model_key,
                ExtraArgs={
                    'ServerSideEncryption': 'AES256'
                }
            )
        
        model_data = f"s3://{bucket}/{model_key}"
        print(f"Model uploaded to: {model_data}")

        # Create PyTorch model with updated configuration
        print("\nCreating PyTorch model...")
        model = PyTorchModel(
            model_data=model_data,
            role=ROLE_ARN,
            framework_version="2.1.0",
            py_version="py310",
            entry_point="inference.py",
            sagemaker_session=session,
            env={
                'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml/model/code',
                'SAGEMAKER_PROGRAM': 'inference.py',
                'PYTHONPATH': '/opt/ml/model/code',
            }
        )

        # Configure serverless config
        serverless_config = ServerlessInferenceConfig(
            memory_size_in_mb=3072,
            max_concurrency=2
        )

        # Deploy model with serverless configuration
        endpoint_name = f"brain-ct-serverless-{int(datetime.utcnow().timestamp())}"
        print(f"\nDeploying serverless endpoint: {endpoint_name}")
        
        predictor = model.deploy(
            endpoint_name=endpoint_name,
            serverless_inference_config=serverless_config
        )

        # Wait for endpoint
        waiter = session.sagemaker_client.get_waiter('endpoint_in_service')
        print("\nWaiting for endpoint deployment...")
        waiter.wait(
            EndpointName=endpoint_name,
            WaiterConfig={
                'Delay': 30,
                'MaxAttempts': 20
            }
        )
        
        # Check status
        endpoint_status = session.sagemaker_client.describe_endpoint(
            EndpointName=endpoint_name
        )
        
        if endpoint_status['EndpointStatus'] == 'InService':
            print("\n✅ Serverless endpoint deployed successfully!")
            print(f"Status: {endpoint_status['EndpointStatus']}")
            print(f"Created: {endpoint_status['CreationTime']}")
            print(f"Endpoint Name: {endpoint_name}")
            print("\nConfiguration:")
            print("✓ PyTorch 1.13.1 with Python 3.9")
            print("✓ Serverless compute configured")
            print(f"✓ Memory size: 3072 MB")
            print(f"✓ Max concurrency: 20")
        else:
            print(f"\n⚠️ Endpoint status: {endpoint_status['EndpointStatus']}")
            
        return predictor
        
    except Exception as e:
        print(f"\n❌ Error during deployment: {str(e)}")
        raise

In [125]:
# Deploy model
predictor = deploy_serverless_model()

[04/27/25 20:39:22] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=273651;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=941686;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/botocore/credentials.py#1352\1352]8;;\


Starting serverless deployment at 2025-04-27 07:49:00 UTC
User: krooldonutz
Role ARN: arn:aws:iam::522814726499:role/SageMakerExecutionRole-krooldonutz
Region: ap-southeast-1
S3 Bucket: sagemaker-ap-southeast-1-522814726499

Preparing model package at 2025-04-27 07:49:00
User: krooldonutz
✓ Model package prepared successfully

Uploading model to S3...
Model uploaded to: s3://sagemaker-ap-southeast-1-522814726499/models/brain-ct/model-1745728791.tar.gz

Creating PyTorch model...

Deploying serverless endpoint: brain-ct-serverless-1745728805


[04/27/25 20:40:05] INFO     Defaulting to CPU type when using serverless inference               ]8;id=590340;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=213829;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/image_uris.py#538\538]8;;\

                    INFO     Repacking model artifact                                                  ]8;id=914559;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/model.py\model.py]8;;\:]8;id=850137;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/model.py#820\820]8;;\
                             (s3://sagemaker-ap-southeast-1-522814726499/models/brain-ct/model-1745728             
                             791.tar.gz), script artifact (None), and dependencies ([]) into single                
                             tar.gz file located at                                                                
                             s3://sagemaker-ap-southeast-1-522814726499/pytorch-inference-2025-04-27-1             
                             2-40-05-240/model.tar.gz. This may take some time depending on model                  
                             size...                                                                               

[04/27/25 20:41:02] INFO     Creating model with name: pytorch-inference-2025-04-27-12-41-02-231    ]8;id=148841;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=181457;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py#4094\4094]8;;\

                    INFO     Creating endpoint-config with name brain-ct-serverless-1745728805      ]8;id=700838;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=198407;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py#6019\6019]8;;\

[04/27/25 20:41:03] INFO     Creating endpoint with name brain-ct-serverless-1745728805             ]8;id=305446;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=180816;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py#4841\4841]8;;\

---!
Waiting for endpoint deployment...

✅ Serverless endpoint deployed successfully!
Status: InService
Created: 2025-04-27 20:41:03.585000+08:00
Endpoint Name: brain-ct-serverless-1745728805

Configuration:
✓ PyTorch 1.13.1 with Python 3.9
✓ Serverless compute configured
✓ Memory size: 3072 MB
✓ Max concurrency: 20


In [128]:
import boto3
import json

def invoke_endpoint(endpoint_name, image_url):
    runtime = boto3.client('sagemaker-runtime')
    
    # Prepare the input
    input_data = {
        "url": image_url
    }
    
    # Invoke endpoint
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps(input_data)
    )
    
    # Parse response
    result = json.loads(response['Body'].read().decode())
    return result

In [130]:

res = invoke_endpoint('brain-ct-serverless-1745728805','https://prod-images-static.radiopaedia.org/images/61513769/9131de5120872bcc1ad19d8198c759904a4b78efa405550c86ec0f19d3dda590_big_gallery.jpeg')

print(res)

{'prediction': 'Non-aneurysm', 'confidence': 0.6151227951049805, 'probabilities': {'non_aneurysm': 0.6151227951049805, 'aneurysm': 0.3848772346973419}, 'metadata': {'timestamp': '2025-04-27 12:47:22', 'user': 'krooldonutz', 'pytorch_version': '2.1.0+cpu'}}


In [ ]:


update_endpoint_config('brain-ct-async-1745714333')